# habitrack

> Cloud-init fragment for the habitrack daily-habit tracker: dedicated system user, git clone + venv, writable data dir, hardened systemd unit

In [ ]:
#| default_exp habitrack

In [ ]:
#| hide
from nbdev.showdoc import *

Single-user FastHTML habit tracker (doyu/habitrack, public repo — see its
DESIGN.md for the app spec). Deploy decisions:

- **Code is pulled, data is not.** `git clone` + `pip install -r requirements.txt`
  from the public repo — no deploy keys (reconcile pattern). The ledger CSV is
  runtime state (DESIGN.md §3): it lives in `/opt/habitrack/app/data`, the unit's
  ONLY writable path, and does NOT survive a rebuild — the rebuild procedure
  copies it off and restores it by hand.
- **public=True.** The phone has no always-on VPN; the app's own session auth
  is the gate (same maturity rationale as reconcile).
- **Port 5002** (5001 is reconcile's). fasthtml's `serve()` reads `PORT` from
  the environment; the unit also pins `TZ=Europe/Helsinki` (belt — the app pins
  the zone in code, DESIGN.md §7).
- Secrets (`HABITRACK_PASSWORD`, `HABITRACK_SECRET_KEY`, `HABITRACK_HTTPS_ONLY=1`)
  live in `/etc/habitrack/env`, installed post-boot; until then the unit stays
  inactive via `ConditionPathExists`. Post-boot order: install env →
  `systemctl start habitrack` → restore `data/ledger.csv` (or let first run
  create the default one).

## `habitrack_service`

In [ ]:
#| export
from boxrecipe.services import check_service, write_file_cmd

_UNIT = """[Unit]
Description=habitrack - single-user daily habit tracker (FastHTML over one CSV)
After=network-online.target
Wants=network-online.target
ConditionPathExists=/etc/habitrack/env
StartLimitIntervalSec=60
StartLimitBurst=3

[Service]
User=habitrack
WorkingDirectory=/opt/habitrack/app
EnvironmentFile=/etc/habitrack/env
Environment=PORT=5002
Environment=TZ=Europe/Helsinki
ExecStart=/opt/habitrack/venv/bin/python /opt/habitrack/app/app.py
Restart=on-failure
RestartSec=5
NoNewPrivileges=yes
ProtectSystem=strict
ProtectHome=yes
ReadWritePaths=/opt/habitrack/app/data
PrivateTmp=yes

[Install]
WantedBy=multi-user.target
"""

def habitrack_service(
)->dict:  # validated service dict for the habitrack tracker (public site + install + unit cmds)
    """Service dict for habitrack: single-user habit tracker, session-auth gated."""
    cmds = [
        "useradd --system --no-create-home --shell /usr/sbin/nologin habitrack",
        "git clone https://github.com/doyu/habitrack.git /opt/habitrack/app",
        "python3 -m venv /opt/habitrack/venv",
        "/opt/habitrack/venv/bin/pip install -r /opt/habitrack/app/requirements.txt",
        # service user reads code+venv via group; X keeps the venv executable
        # whatever umask the surrounding cloud-init script runs under
        "chown -R root:habitrack /opt/habitrack",
        "chmod -R u=rwX,g=rX,o= /opt/habitrack",
        # the unit's ONLY writable path: the ledger is runtime state (DESIGN.md §3)
        "install -d -m 770 -o habitrack -g habitrack /opt/habitrack/app/data",
        "install -d -m 700 -o root -g root /etc/habitrack",
        write_file_cmd("/etc/systemd/system/habitrack.service", _UNIT),
        "systemctl daemon-reload",
        "systemctl enable habitrack",
    ]
    # public=True: the phone has no always-on VPN — the app's session auth is the gate
    return check_service({"name": "habitrack", "domain": "habitrack.ninjalabo.ai",
                          "port": 5002, "public": True,
                          "packages": ["python3-venv"], "cmds": cmds})

In [ ]:
svc = habitrack_service()
assert svc["name"] == "habitrack" and svc["domain"] == "habitrack.ninjalabo.ai"
# public: checked from the phone anywhere — the app's session auth is the gate
assert svc["port"] == 5002 and svc["public"] is True
assert "python3-venv" in svc["packages"]

joined = "\n".join(svc["cmds"])
# system user, no home: the app owns nothing but its data dir
assert "useradd --system --no-create-home --shell /usr/sbin/nologin habitrack" in joined
# code is pulled from the public repo — no deploy keys (reconcile pattern)
assert "git clone https://github.com/doyu/habitrack.git /opt/habitrack/app" in joined
assert "python3 -m venv /opt/habitrack/venv" in joined
assert "/opt/habitrack/venv/bin/pip install -r /opt/habitrack/app/requirements.txt" in joined
# service user reads code+venv via group; other users get nothing
assert "chown -R root:habitrack /opt/habitrack" in joined
assert "chmod -R u=rwX,g=rX,o= /opt/habitrack" in joined
assert joined.index("git clone") < joined.index("chmod -R u=rwX")
# data dir AFTER the blanket chown/chmod, or its owner would be clobbered
assert "install -d -m 770 -o habitrack -g habitrack /opt/habitrack/app/data" in joined
assert joined.index("chmod -R u=rwX") < joined.index("install -d -m 770 -o habitrack")
# secrets dir (root-only) so the post-boot `install -m 600` has a target
assert "install -d -m 700 -o root -g root /etc/habitrack" in joined
# the unit: identity, wait-state, network ordering, hardening, restart policy
for token in ("/etc/systemd/system/habitrack.service",
              "User=habitrack",
              "WorkingDirectory=/opt/habitrack/app",
              "EnvironmentFile=/etc/habitrack/env",
              "ConditionPathExists=/etc/habitrack/env",
              "Environment=PORT=5002", "Environment=TZ=Europe/Helsinki",
              "After=network-online.target", "Wants=network-online.target",
              "ExecStart=/opt/habitrack/venv/bin/python /opt/habitrack/app/app.py",
              "Restart=on-failure", "RestartSec=5",
              "StartLimitIntervalSec=60", "StartLimitBurst=3",
              "NoNewPrivileges=yes", "ProtectSystem=strict",
              "ProtectHome=yes", "PrivateTmp=yes",
              "ReadWritePaths=/opt/habitrack/app/data",
              "WantedBy=multi-user.target"):
    assert token in joined, token
assert "systemctl daemon-reload" in joined
assert "systemctl enable habitrack" in joined
# never a secret name/value in the fragment — env values arrive post-boot
for name in ("HABITRACK_PASSWORD", "HABITRACK_SECRET_KEY", "HABITRACK_HTTPS_ONLY"):
    assert name not in joined, name

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()